# Test legal chunking logic

Notebook này dùng để test nhanh logic tách văn bản pháp luật thành `legal_units` và `chunks` trước khi đưa vào `src/`.

In [ ]:
from pathlib import Path
import json
import re
from pprint import pprint

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED_DIR = ROOT / "data" / "processed"
STRUCTURED_DIR = ROOT / "data" / "structured"
CHUNKS_DIR = ROOT / "data" / "chunks"

STRUCTURED_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT)
print(sorted(p.name for p in PROCESSED_DIR.glob("*.json")))

d:\season2_3rd\vietnam-proptech-rag
['101_2024_ND_CP.json', '102_2024_ND_CP.json', '27_2023_QH15.json', '29_2023_QH15.json', '31_2024_QH15.json']


## Load một văn bản mẫu

In [ ]:
DOC_ID = "31_2024_QH15"
doc_path = PROCESSED_DIR / f"{DOC_ID}.json"

with doc_path.open("r", encoding="utf-8") as f:
    doc = json.load(f)

text = doc["clean_text"]
print(doc["title"])
print(doc["doc_number"])
print(len(text), "chars")
print(text[:700])

Luật Đất đai 2024
31/2024/QH15
527010 chars
QUỐC HỘI -------
QUỐC HỘI -------
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc ---------------
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc ---------------
Luật số: 31/2024/QH15
Luật số: 31/2024/QH15
Hà Nội, ngày 18 tháng 01 năm 2024
Hà Nội, ngày 18 tháng 01 năm 2024
LUẬT
ĐẤT ĐAI
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam ;
Quốc hội ban hành Luật Đất đai.

# Chương I

QUY ĐỊNH CHUNG

## Điều 1. Phạm vi điều chỉnh

Luật này quy định về chế độ sở hữu đất đai, quyền hạn và trách nhiệm của Nhà nước đại diện chủ sở hữu toàn dân về đất đai và thống nhất quản lý về đất đai, chế độ quản lý và sử dụng đất đai, quyền và nghĩa vụ của công dân, ngư
4. Ủy


## Helper: bỏ dòng trùng liền kề

Do `clean_dataset.py` đang lấy cả `td` và `p`, một số dòng đầu văn bản có thể bị lặp. Cell này chỉ xử lý lặp liền kề để test chunking sạch hơn.

In [ ]:
def dedupe_adjacent_lines(text: str) -> str:
    lines = [line.strip() for line in text.splitlines()]
    kept = []
    previous = None
    for line in lines:
        if not line:
            if kept and kept[-1] != "":
                kept.append("")
            previous = None
            continue
        if line != previous:
            kept.append(line)
        previous = line
    return "\n".join(kept).strip()

clean_text = dedupe_adjacent_lines(text)
print(clean_text[:700])

QUỐC HỘI -------
CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc ---------------
Luật số: 31/2024/QH15
Hà Nội, ngày 18 tháng 01 năm 2024
LUẬT
ĐẤT ĐAI
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam ;
Quốc hội ban hành Luật Đất đai.

# Chương I

QUY ĐỊNH CHUNG

## Điều 1. Phạm vi điều chỉnh

Luật này quy định về chế độ sở hữu đất đai, quyền hạn và trách nhiệm của Nhà nước đại diện chủ sở hữu toàn dân về đất đai và thống nhất quản lý về đất đai, chế độ quản lý và sử dụng đất đai, quyền và nghĩa vụ của công dân, ngư
4. Ủy ban nhân dân cấp tỉnh phê duyệt quy hoạch sử dụng đất cấp huyện, kế hoạch sử dụng đấời sử dụng đất đối với đất đai thuộc lãnh thổ của nước Cộng hòa xã 


## Parse theo Điều

Mục tiêu là tạo đơn vị pháp lý đủ citation, chưa cần tối ưu tuyệt đối. Sau khi test ổn mới đưa vào `src/legal_parser/`.

In [ ]:
ARTICLE_RE = re.compile(r"(?m)^##\s+Điều\s+(\d+)\.\s*(.+?)\s*$")
CHAPTER_RE = re.compile(r"(?m)^#\s+Chương\s+([0-9IVXLCDM]+)\s*$", re.IGNORECASE)

def current_chapter(text_before_article: str):
    matches = list(CHAPTER_RE.finditer(text_before_article))
    if not matches:
        return None
    return matches[-1].group(1)

def parse_articles(doc: dict, text: str):
    matches = list(ARTICLE_RE.finditer(text))
    articles = []

    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(text)
        article_no = int(match.group(1))
        article_title = match.group(2).strip()
        body = text[match.end():end].strip()
        chapter = current_chapter(text[:start])

        articles.append({
            "unit_id": f"{doc['doc_id']}__dieu_{article_no}",
            "doc_id": doc["doc_id"],
            "doc_title": doc["title"],
            "doc_number": doc["doc_number"],
            "doc_type": doc["doc_type"],
            "source_url": doc["source_url"],
            "chapter": chapter,
            "article_no": article_no,
            "article_title": article_title,
            "text": body,
            "citation": f"{doc['title']}, Điều {article_no}",
        })

    return articles

articles = parse_articles(doc, clean_text)
print(len(articles), "articles")
pprint(articles[0] | {"text": articles[0]["text"][:500]})

259 articles
{'article_no': 1,
 'article_title': 'Phạm vi điều chỉnh',
 'chapter': 'I',
 'citation': 'Luật Đất đai 2024, Điều 1',
 'doc_id': '31_2024_QH15',
 'doc_number': '31/2024/QH15',
 'doc_title': 'Luật Đất đai 2024',
 'doc_type': 'luat',
 'source_url': 'https://thuvienphapluat.vn/van-ban/Bat-dong-san/Luat-Dat-dai-2024-31-2024-QH15-523642.aspx',
 'text': 'Luật này quy định về chế độ sở hữu đất đai, quyền hạn và trách nhiệm '
         'của Nhà nước đại diện chủ sở hữu toàn dân về đất đai và thống nhất '
         'quản lý về đất đai, chế độ quản lý và sử dụng đất đai, quyền và '
         'nghĩa vụ của công dân, ngư\n'
         '4. Ủy ban nhân dân cấp tỉnh phê duyệt quy hoạch sử dụng đất cấp '
         'huyện, kế hoạch sử dụng đấời sử dụng đất đối với đất đai thuộc lãnh '
         'thổ của nước Cộng hòa xã hội chủ nghĩa Việt Nam.',
 'unit_id': '31_2024_QH15__dieu_1'}


## Parse Khoản trong từng Điều

Nếu Điều dài, tách thành khoản. Nếu không tìm được khoản, giữ nguyên Điều.

In [ ]:
CLAUSE_RE = re.compile(r"(?m)^(\d+)\s*\.\s+")

def is_inside_quote(text: str, pos: int) -> bool:
    prefix = text[:pos]
    return prefix.count("“") > prefix.count("”")

def parse_clauses(article: dict):
    body = article["text"]
    matches = [m for m in CLAUSE_RE.finditer(body) if not is_inside_quote(body, m.start())]
    if not matches:
        return [{**article, "level": "article"}]
    if int(matches[0].group(1)) != 1 and body[:matches[0].start()].strip():
        return [{**article, "level": "article", "parse_warning": "first_clause_is_not_1"}]

    clauses = []
    intro = body[:matches[0].start()].strip()

    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(body)
        clause_no = int(match.group(1))
        clause_text = body[start:end].strip()
        if intro:
            clause_text = intro + "\n" + clause_text

        clauses.append({
            **{k: v for k, v in article.items() if k != "text"},
            "unit_id": f"{article['unit_id']}__khoan_{clause_no}",
            "level": "clause",
            "clause_no": clause_no,
            "text": clause_text,
            "citation": f"{article['citation']} khoản {clause_no}",
        })

    return clauses

legal_units = []
for article in articles:
    legal_units.extend(parse_clauses(article))

print(len(legal_units), "legal units")
pprint(legal_units[:3])

1256 legal units
[{'article_no': 1,
  'article_title': 'Phạm vi điều chỉnh',
  'chapter': 'I',
  'citation': 'Luật Đất đai 2024, Điều 1 khoản 4',
  'clause_no': 4,
  'doc_id': '31_2024_QH15',
  'doc_number': '31/2024/QH15',
  'doc_title': 'Luật Đất đai 2024',
  'doc_type': 'luat',
  'level': 'clause',
  'source_url': 'https://thuvienphapluat.vn/van-ban/Bat-dong-san/Luat-Dat-dai-2024-31-2024-QH15-523642.aspx',
  'text': 'Luật này quy định về chế độ sở hữu đất đai, quyền hạn và trách '
          'nhiệm của Nhà nước đại diện chủ sở hữu toàn dân về đất đai và thống '
          'nhất quản lý về đất đai, chế độ quản lý và sử dụng đất đai, quyền '
          'và nghĩa vụ của công dân, ngư\n'
          '4. Ủy ban nhân dân cấp tỉnh phê duyệt quy hoạch sử dụng đất cấp '
          'huyện, kế hoạch sử dụng đấời sử dụng đất đối với đất đai thuộc lãnh '
          'thổ của nước Cộng hòa xã hội chủ nghĩa Việt Nam.',
  'unit_id': '31_2024_QH15__dieu_1__khoan_4'},
 {'article_no': 2,
  'article_title': 'Đ

## Build chunks cho RAG

Text dùng để embed nên có header ngữ cảnh để model hiểu chunk này thuộc văn bản/Điều/Khoản nào.

In [ ]:
MAX_CHARS = 2200

def split_long_text(text: str, max_chars: int = MAX_CHARS):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    current = []
    current_len = 0

    for paragraph in paragraphs:
        if current and current_len + len(paragraph) + 1 > max_chars:
            chunks.append("\n".join(current))
            current = []
            current_len = 0
        current.append(paragraph)
        current_len += len(paragraph) + 1

    if current:
        chunks.append("\n".join(current))

    return chunks

def make_chunk_text(unit: dict, body: str):
    header = f"{unit['doc_title']} | Điều {unit['article_no']}. {unit['article_title']}"
    if unit.get("clause_no"):
        header += f" | Khoản {unit['clause_no']}"
    return f"{header}\n\n{body}".strip()

def build_chunks(units):
    chunks = []
    for unit in units:
        parts = split_long_text(unit["text"])
        for idx, part in enumerate(parts, 1):
            chunk_id = unit["unit_id"] if len(parts) == 1 else f"{unit['unit_id']}__part_{idx}"
            chunks.append({
                "chunk_id": chunk_id,
                "text": make_chunk_text(unit, part),
                "metadata": {k: v for k, v in unit.items() if k != "text"},
            })
    return chunks

chunks = build_chunks(legal_units)
print(len(chunks), "chunks")
print(chunks[0]["text"][:900])
pprint(chunks[0]["metadata"])

1270 chunks
Luật Đất đai 2024 | Điều 1. Phạm vi điều chỉnh | Khoản 4

Luật này quy định về chế độ sở hữu đất đai, quyền hạn và trách nhiệm của Nhà nước đại diện chủ sở hữu toàn dân về đất đai và thống nhất quản lý về đất đai, chế độ quản lý và sử dụng đất đai, quyền và nghĩa vụ của công dân, ngư
4. Ủy ban nhân dân cấp tỉnh phê duyệt quy hoạch sử dụng đất cấp huyện, kế hoạch sử dụng đấời sử dụng đất đối với đất đai thuộc lãnh thổ của nước Cộng hòa xã hội chủ nghĩa Việt Nam.
{'article_no': 1,
 'article_title': 'Phạm vi điều chỉnh',
 'chapter': 'I',
 'citation': 'Luật Đất đai 2024, Điều 1 khoản 4',
 'clause_no': 4,
 'doc_id': '31_2024_QH15',
 'doc_number': '31/2024/QH15',
 'doc_title': 'Luật Đất đai 2024',
 'doc_type': 'luat',
 'level': 'clause',
 'source_url': 'https://thuvienphapluat.vn/van-ban/Bat-dong-san/Luat-Dat-dai-2024-31-2024-QH15-523642.aspx',
 'unit_id': '31_2024_QH15__dieu_1__khoan_4'}


## Kiểm tra nhanh chất lượng chunk

In [ ]:
lengths = [len(c["text"]) for c in chunks]
print("min:", min(lengths))
print("max:", max(lengths))
print("avg:", round(sum(lengths) / len(lengths), 1))
print("too long:", sum(1 for x in lengths if x > MAX_CHARS + 300))
print("missing citation:", sum(1 for c in chunks if not c["metadata"].get("citation")))

for c in chunks[:5]:
    print("-", c["chunk_id"], "|", c["metadata"]["citation"], "|", len(c["text"]))

min: 78
max: 2432
avg: 528.1
too long: 0
missing citation: 0
- 31_2024_QH15__dieu_1__khoan_4 | Luật Đất đai 2024, Điều 1 khoản 4 | 465
- 31_2024_QH15__dieu_2__khoan_1 | Luật Đất đai 2024, Điều 2 khoản 1 | 211
- 31_2024_QH15__dieu_2__khoan_2 | Luật Đất đai 2024, Điều 2 khoản 2 | 78
- 31_2024_QH15__dieu_2__khoan_3 | Luật Đất đai 2024, Điều 2 khoản 3 | 126
- 31_2024_QH15__dieu_3__khoan_1 | Luật Đất đai 2024, Điều 3 khoản 1 | 365


## Lưu output thử nghiệm

In [ ]:
units_path = STRUCTURED_DIR / f"{DOC_ID}_legal_units.jsonl"
chunks_path = CHUNKS_DIR / f"{DOC_ID}_chunks.jsonl"

with units_path.open("w", encoding="utf-8") as f:
    for item in legal_units:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with chunks_path.open("w", encoding="utf-8") as f:
    for item in chunks:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(units_path)
print(chunks_path)